# 03 特征工程与指标筛选
**论文章节对应**：第3章 指标体系构建与特征工程

本notebook完成以下任务：
1. 衍生特征构造（F1单层面积、F2是否高层）
2. 目标变量log变换
3. Pearson相关性分析（图3-1）
4. VIF多重共线性检验（图3-2）
5. Pipeline预处理器构建与验证

**输入**：`data/processed/housing_cost_clean.csv`
**输出**：图3-1/3-2 + Pipeline对象（供notebook 04使用）

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from src.preprocessing import (
    build_preprocessor, add_derived_features,
    log_transform_target, log_transform_area,
    time_based_split, year_expanding_cv,
    NUMERIC_FEATURES, CATEGORICAL_FEATURES, ORDINAL_FEATURES, BINARY_FEATURES, TARGET
)
from src.feature_engineering import (
    plot_correlation_heatmap, check_vif, plot_vif_barh
)

matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# 读取清洗后数据
df = pd.read_csv('data/processed/housing_cost_clean.csv', encoding='utf-8-sig')
print(f'读取数据: {df.shape}')
df.head(3)

## 3.1 衍生特征构造
- **F1 floor_density**（单层平均面积）= total_area / above_floors
- **F2 is_high_rise**（是否高层）= 1 if above_floors ≥ 12 else 0

In [ ]:
# 添加衍生特征
df = add_derived_features(df)
print(f'F1 floor_density 统计：\n{df["floor_density"].describe().round(1)}')
print(f'\nF2 is_high_rise 分布：')
print(df['is_high_rise'].value_counts())
print(f'  高层占比: {df["is_high_rise"].mean():.1%}')

## 3.2 目标变量与面积log变换

In [ ]:
# 目标变量log1p变换（减小右偏影响，使残差更接近正态）
df = log_transform_target(df, target_col='unit_cost')
print(f'目标变量 unit_cost_log 统计：\n{df["unit_cost_log"].describe().round(4)}')

# 总建筑面积log1p变换（同样存在右偏）
df = log_transform_area(df, col='total_area')
print(f'\ntotal_area（log后）统计：\n{df["total_area"].describe().round(4)}')

## 3.3 Pearson相关性热力图（图3-1）

In [ ]:
# 数值特征相关性分析
numeric_cols_for_corr = NUMERIC_FEATURES + ORDINAL_FEATURES + BINARY_FEATURES
# 只保留数据集中存在的列
numeric_cols_for_corr = [c for c in numeric_cols_for_corr if c in df.columns]

corr = plot_correlation_heatmap(
    df,
    numeric_cols=numeric_cols_for_corr,
    target_col='unit_cost_log',
    save_path='outputs/figures/fig3_1_correlation.png'
)

## 3.4 VIF多重共线性检验（图3-2）

In [ ]:
# VIF仅适用于数值特征
vif_cols = [c for c in NUMERIC_FEATURES + ORDINAL_FEATURES + BINARY_FEATURES
            if c in df.columns]

X_numeric = df[vif_cols].select_dtypes(include=[np.number])

vif_data = check_vif(X_numeric, threshold=10)

In [ ]:
# 绘制VIF柱状图（图3-2）
plot_vif_barh(
    vif_data,
    threshold=10,
    save_path='outputs/figures/fig3_2_vif.png'
)

In [ ]:
# 根据VIF结果决定是否剔除特征
severe_vif = vif_data[vif_data['VIF'] > 10]
if not severe_vif.empty:
    print(f'⚠ 以下特征VIF>10，考虑剔除：{severe_vif["feature"].tolist()}')
    print('  建议：检查是否与其他特征高度相关，必要时删除或合并')
    # 从数值特征列表中移除VIF过高的特征
    drop_features = severe_vif['feature'].tolist()
    NUMERIC_FEATURES_FINAL = [f for f in NUMERIC_FEATURES if f not in drop_features]
    print(f'  最终数值特征: {NUMERIC_FEATURES_FINAL}')
else:
    NUMERIC_FEATURES_FINAL = NUMERIC_FEATURES
    print('✓ 所有特征VIF均合格，无需剔除')

## 3.5 构建Pipeline预处理器

In [ ]:
# 确认数据集中存在的分类特征
cat_features_final = [c for c in CATEGORICAL_FEATURES if c in df.columns]
ord_features_final = [c for c in ORDINAL_FEATURES if c in df.columns]
bin_features_final = [c for c in BINARY_FEATURES if c in df.columns]
num_features_final = [c for c in NUMERIC_FEATURES_FINAL if c in df.columns]

print('特征分组：')
print(f'  数值特征({len(num_features_final)}个): {num_features_final}')
print(f'  分类特征({len(cat_features_final)}个): {cat_features_final}')
print(f'  有序特征({len(ord_features_final)}个): {ord_features_final}')
print(f'  二元特征({len(bin_features_final)}个): {bin_features_final}')

# 构建预处理Pipeline
preprocessor = build_preprocessor(
    numeric_features=num_features_final,
    categorical_features=cat_features_final,
    ordinal_features=ord_features_final,
    binary_features=bin_features_final,
)
print('\n✓ 预处理Pipeline构建完成')
print(preprocessor)

## 3.6 时序划分 & Pipeline验证

In [ ]:
# 时序划分
train_df, test_df = time_based_split(df, test_year=2025)

all_feature_cols = num_features_final + cat_features_final + ord_features_final + bin_features_final

X_train = train_df[all_feature_cols]
y_train = train_df[TARGET]
X_test  = test_df[all_feature_cols]
y_test  = test_df[TARGET]

print(f'\n训练集: X={X_train.shape}, y={y_train.shape}')
print(f'测试集: X={X_test.shape}, y={y_test.shape}')

In [ ]:
# 验证Pipeline（只在训练集fit）
preprocessor.fit(X_train)
X_train_proc = preprocessor.transform(X_train)   # fit后的训练集
X_test_proc  = preprocessor.transform(X_test)    # 只transform，不fit

print(f'预处理后训练集维度: {X_train_proc.shape}')
print(f'预处理后测试集维度: {X_test_proc.shape}')
print('✓ 预处理Pipeline验证通过')

In [ ]:
# 打印Expanding Window CV分割情况
print('\nExpanding Window CV分割：')
cv_splits = list(year_expanding_cv(train_df))
print(f'总共 {len(cv_splits)} 个Fold')

In [ ]:
# 保存预处理配置供后续notebook使用
config = {
    'num_features': num_features_final,
    'cat_features': cat_features_final,
    'ord_features': ord_features_final,
    'bin_features': bin_features_final,
    'all_features': all_feature_cols,
    'target': TARGET,
}
joblib.dump(config, 'outputs/saved_models/feature_config.pkl')
joblib.dump(preprocessor, 'outputs/saved_models/preprocessor.pkl')

# 保存划分后数据
train_df.to_csv('data/processed/train.csv', index=False, encoding='utf-8-sig')
test_df.to_csv('data/processed/test.csv', index=False, encoding='utf-8-sig')

print('✓ 特征配置已保存 → outputs/saved_models/feature_config.pkl')
print('✓ 预处理器已保存 → outputs/saved_models/preprocessor.pkl')
print('✓ 训练集已保存  → data/processed/train.csv')
print('✓ 测试集已保存  → data/processed/test.csv')